<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
### <center> المؤلف: تاتيانا كوداسوفا، ODS Slack @kudasova
    
## <center> البرنامج التعليمي
## <center> التحقق المتبادل المتداخل
    



### لماذا التحقق المتبادل المتداخل؟
في كثير من الأحيان نريد ضبط معلمات النموذج. أي أننا نريد العثور على قيمة المعلمة التي تقلل دالة الخسارة لدينا. وأفضل طريقة للقيام بذلك، كما نعلم بالفعل، هي التحقق المتبادل.
ومع ذلك، كما أشار كاولي وتالبوت في [ورقتهم البحثية لعام 2010](http://jmlr.org/papers/volume11/cawley10a/cawley10a.pdf)، نظرًا لأننا استخدمنا مجموعة الاختبار لتحديد قيم المعلمة وتقييم النموذج، فإننا نخاطر بتحيز تقييمات النموذج بشكل متفائل. لهذا السبب، إذا تم استخدام مجموعة اختبار لتحديد معلمات النموذج، فإننا نحتاج إلى مجموعة اختبار مختلفة للحصول على تقييم غير متحيز لهذا النموذج المحدد. بشكل أساسي، يمكننا التفكير في اختيار النموذج كإجراء تدريبي آخر، وبالتالي، سنحتاج إلى مجموعة اختبار مستقلة ذات حجم مناسب لم نرها من قبل للحصول على تقدير غير متحيز لأداء النماذج. في كثير من الأحيان، هذا ليس في المتناول. إحدى الطرق الجيدة للتغلب على هذه المشكلة هي استخدام التحقق المتبادل المتداخل.



### شرح التحقق المتبادل المتداخل
يحتوي التحقق المتبادل المتداخل على التحقق المتبادل الداخلي المتداخل في التحقق المتبادل الخارجي. أولاً، يتم استخدام التحقق المتبادل الداخلي لضبط المعلمات واختيار النموذج الأفضل. ثانيًا، يتم استخدام التحقق المتبادل الخارجي لتقييم النموذج المحدد بواسطة التحقق المتبادل الداخلي.
<img src="../../img/nestedCV.png" width="800"/>
تخيل أن لدينا نماذج _N_ ونريد استخدام التحقق المتبادل الداخلي _L_fold لضبط المعلمات الفائقة والتحقق المتبادل الخارجي K-fold لتقييم النماذج. ثم الخوارزمية هي كما يلي:1. قم بتقسيم مجموعة البيانات إلى طيات التحقق المتبادل _K_ بشكل عشوائي.
2. لكل طية _k=1,2,...,K_: (الحلقة الخارجية لتقييم النموذج باستخدام المعلمة الفائقة المحددة) <br>
  2.1. دع `test` يتم طيه _k_ <br>
  2.2. اجعل `trainval` جميع البيانات باستثناء تلك الموجودة في الطية _k_ <br>
  2.3. قم بتقسيم `trainval` بشكل عشوائي إلى _L_ طيات <br>
  2.4. لكل طية _l=1,2,...L_: (الحلقة الداخلية لضبط المعلمة الفائقة) <br>
> 2.4.1 اسمح بطي `val` _l_ <br>
> 2.4.2 اجعل `train` جميع البيانات باستثناء تلك الموجودة في `test` أو `val` <br>
> 2.4.3 تدريب كل نموذج من نماذج _N_ مع كل معلمة تشعبية على `train`، وتقييمها على `val`. تتبع مقاييس الأداء <br> 
  2.5. بالنسبة لكل إعداد للمعلمة التشعبية، احسب متوسط ​​نقاط المقاييس عبر الطيات _L_، واختر أفضل إعداد للمعلمة التشعبية. <br>
  2.6. قم بتدريب كل نموذج من نماذج _N_ باستخدام أفضل معلمة تشعبية على `trainval`. قم بتقييم أدائها على `test` واحفظ النتيجة لـfold _k_ <br>
  
3. بالنسبة لكل نموذج من نماذج _N_، قم بحساب متوسط الدرجة على جميع طيات _K_، وقم بالإبلاغ عنها كخطأ في التعميم.
في الصورة أعلاه والرمز أدناه اخترنا _L = 2_ و _K = 5_، ولكن يمكنك اختيار أرقام مختلفة.



### التنفيذ


In [ ]:
# Load required packages
import numpy as np
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


بيانات هذا البرنامج التعليمي هي [بيانات سرطان الثدي](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) مع 30 ميزة ومتغير هدف ثنائي.


In [ ]:
# Load the data
dataset = datasets.load_breast_cancer()

# Create X from the features
X = dataset.data

# Create y from the target
y = dataset.target

In [ ]:
# Making train set for Nested CV and test set for final model evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=1, stratify=y
)

# Initializing Classifiers
clf1 = LogisticRegression(solver="liblinear", random_state=1)
clf2 = KNeighborsClassifier()
clf3 = DecisionTreeClassifier(random_state=1)
clf4 = SVC(kernel="rbf", random_state=1)

# Building the pipelines
pipe1 = Pipeline([("std", StandardScaler()), ("clf1", clf1)])

pipe2 = Pipeline([("std", StandardScaler()), ("clf2", clf2)])

pipe4 = Pipeline([("std", StandardScaler()), ("clf4", clf4)])


# Setting up the parameter grids
param_grid1 = [
    {"clf1__penalty": ["l1", "l2"], "clf1__C": np.power(10.0, np.arange(-4, 4))}
]

param_grid2 = [{"clf2__n_neighbors": list(range(1, 10)), "clf2__p": [1, 2]}]

param_grid3 = [
    {"max_depth": list(range(1, 10)) + [None], "criterion": ["gini", "entropy"]}
]

param_grid4 = [
    {
        "clf4__C": np.power(10.0, np.arange(-4, 4)),
        "clf4__gamma": np.power(10.0, np.arange(-5, 0)),
    }
]

# Setting up multiple GridSearchCV objects as inner CV, 1 for each algorithm
gridcvs = {}
inner_cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=1)

for pgrid, est, name in zip(
    (param_grid1, param_grid2, param_grid3, param_grid4),
    (pipe1, pipe2, clf3, pipe4),
    ("Logit", "KNN", "DTree", "SVM"),
):
    gcv = GridSearchCV(
        estimator=est,
        param_grid=pgrid,
        scoring="accuracy",
        n_jobs=1,
        cv=inner_cv,
        verbose=0,
        refit=True,
    )
    gridcvs[name] = gcv

In [ ]:
# Making an outer CV
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

for name, gs_est in sorted(gridcvs.items()):
    nested_score = cross_val_score(gs_est, X=X_train, y=y_train, cv=outer_cv, n_jobs=1)
    print(
        "%s | outer ACC %.2f%% +/- %.2f"
        % (name, nested_score.mean() * 100, nested_score.std() * 100)
    )

In [ ]:
# Fitting a model to the whole training set using the "best" algorithm
best_algo = gridcvs["SVM"]

best_algo.fit(X_train, y_train)
train_acc = accuracy_score(y_true=y_train, y_pred=best_algo.predict(X_train))
test_acc = accuracy_score(y_true=y_test, y_pred=best_algo.predict(X_test))

print("Accuracy %.2f%% (average over CV train folds)" % (100 * best_algo.best_score_))
print("Best Parameters: %s" % gridcvs["SVM"].best_params_)
print("Training Accuracy: %.2f%%" % (100 * train_acc))
print("Test Accuracy: %.2f%%" % (100 * test_acc))


### الخلاصة
تعلمنا في هذا البرنامج التعليمي كيفية استخدام التحقق المتبادل المتداخل لضبط المعلمات الفائقة وتقييم النموذج. نأمل أن يساعدك ذلك في مسابقات Kaggle أو مشاريع ML الخاصة بك.عند كتابة هذا البرنامج التعليمي استخدمنا المصادر التالية:
1. [مقالة سيباستيان راشكا](https://sebastianraschka.com/blog/2018/model-evaluation-selection-part4.html)
2. [وكذلك الكود الخاص به من GitHub](https://github.com/rasbt/model-eval-article-supplementary/blob/master/code/nested_cv_code.ipynb.)
3. [مقالة وينا جين](https://weina.me/nested-cross-validation/)